# Operating on tensors

_PyTorch (a complete course)_

**Add, multiply, broadcast, matmul, reduce, and reshape — the moves every model is built from.**

A tensor is just an n-dimensional grid of numbers with a shape (its size along each axis, e.g. (batch, features)).

       Almost everything you do is one of five verbs: apply a function to each number, line two tensors up and combine them, multiply matrices, collapse an axis, or rearrange the grid.

---

This notebook is a practice scaffold for the **Operating on tensors** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch

torch.manual_seed(0)

# ---- 1. BROADCASTING: (3,1) + (1,4) -> (3,4) ----
col = torch.tensor([[10.], [20.], [30.]])   # shape (3,1) -- a column
row = torch.tensor([[1., 2., 3., 4.]])      # shape (1,4) -- a row
grid = col + row                            # broadcasts: col repeats across cols, row down rows
print("col", tuple(col.shape), " row", tuple(row.shape), " grid", tuple(grid.shape))
print(grid)                                 # top-left 11, bottom-right 34

# ---- 2. ELEMENTWISE * vs MATRIX MULTIPLY @ ----
A = torch.arange(6.).reshape(2, 3)          # (2,3)
B = torch.ones(2, 3)                        # (2,3)
print("A * B (elementwise) ->", tuple((A * B).shape))   # (2,3): number-by-number
C = torch.ones(3, 2)                        # (3,2)
print("A @ C (matmul)      ->", tuple((A @ C).shape))   # (2,2): inner 3 cancels
# torch.matmul is the same as @; bmm does it per batch element:
batch = torch.randn(8, 2, 3)               # 8 matrices of (2,3)
w     = torch.randn(8, 3, 5)               # 8 matrices of (3,5)
print("bmm batch           ->", tuple(torch.bmm(batch, w).shape))  # (8,2,5)

# ---- 3. REDUCTION over a chosen dim, with / without keepdim ----
row_sum      = grid.sum(dim=1)                  # collapse columns -> (3,)
row_sum_keep = grid.sum(dim=1, keepdim=True)    # keep axis as 1 -> (3,1)
print("sum dim=1          ->", tuple(row_sum.shape), row_sum.tolist())
print("sum dim=1 keepdim  ->", tuple(row_sum_keep.shape))
# keepdim lets it broadcast back (this is exactly the softmax-normalize trick):
normalized = grid / row_sum_keep                # (3,4) / (3,1) -> (3,4), rows sum to 1
print("each row sums to 1 ->", normalized.sum(dim=1).tolist())
print("argmax over dim=1  ->", grid.argmax(dim=1).tolist())  # index of max per row

# ---- 4. RESHAPING: view / reshape / permute / unsqueeze ----
x = torch.arange(24).reshape(2, 3, 4)           # (2,3,4)
print("x                  ->", tuple(x.shape))
print("x.view(2, 12)      ->", tuple(x.view(2, 12).shape))      # contiguous: view ok
xp = x.permute(0, 2, 1)                          # (2,4,3): now non-contiguous
print("x.permute(0,2,1)   ->", tuple(xp.shape), "contiguous?", xp.is_contiguous())
try:
    xp.view(-1)                                  # fails: non-contiguous
except RuntimeError as e:
    print("xp.view(-1) FAILED -> use reshape/contiguous")
print("xp.reshape(-1)     ->", tuple(xp.reshape(-1).shape))     # (24,): reshape copies
print("x.flatten(1)       ->", tuple(x.flatten(1).shape))       # (2,12): merge last two axes
print("x.unsqueeze(0)     ->", tuple(x.unsqueeze(0).shape))     # (1,2,3,4): add batch axis
print("cat dim=0          ->", tuple(torch.cat([x, x], dim=0).shape))   # (4,3,4)
print("stack dim=0        ->", tuple(torch.stack([x, x], dim=0).shape)) # (2,2,3,4) new axis

## Visualize the data & results

_When a (3,1) column broadcasts against a (1,4) row, what (3,4) grid comes out — and what do the per-row sums look like?_

In [ ]:
import numpy as np

col = np.array([[10], [20], [30]])   # shape (3,1) -- a column
row = np.array([[1, 2, 3, 4]])       # shape (1,4) -- a row

grid = col + row                     # broadcasts to (3,4); matches torch exactly
print("grid shape", grid.shape)
print(grid)
# [[11 12 13 14]
#  [21 22 23 24]
#  [31 32 33 34]]

row_sum = grid.sum(axis=1)           # reduce over columns -> (3,)
print("row sums", row_sum)           # [ 50  90 130]

# row_sum keeping the axis (the softmax-normalize trick):
row_sum_keep = grid.sum(axis=1, keepdims=True)  # shape (3,1)
print("keepdims shape", row_sum_keep.shape)     # (3, 1)

# --- the two charts ---
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
im = ax[0].imshow(grid, cmap="viridis")
ax[0].set_xticks(range(4)); ax[0].set_xticklabels([1, 2, 3, 4])
ax[0].set_yticks(range(3)); ax[0].set_yticklabels([10, 20, 30])
ax[0].set_title("(3,1)+(1,4) -> (3,4)")
for i in range(3):
    for j in range(4):
        ax[0].text(j, i, grid[i, j], ha="center", va="center", color="w")
ax[1].bar(["row 0", "row 1", "row 2"], row_sum,
          color=["#4ea1ff", "#7ee787", "#c89bff"])
ax[1].set_title("grid.sum(dim=1) = [50, 90, 130]")
plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Type this in Colab. Make A = torch.arange(6.).reshape(2, 3) and B = torch.ones(2, 3). Compute the elementwise product A * B and print its shape. Then make C = torch.ones(3, 2) and compute the matrix product A @ C and print its shape. Predict both shapes before running.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use * for elementwise and @ for matmul. — _They are different ops: * keeps the shape, @ follows the matmul shape rule._
- Apply (2,3) @ (3,2): the inner 3 cancels. — _Inner dims must match; the outer dims (2,2) survive._

**Answer:** A = torch.arange(6.).reshape(2, 3)
B = torch.ones(2, 3)
C = torch.ones(3, 2)
print((A * B).shape)   # torch.Size([2, 3])  elementwise keeps shape
print((A @ C).shape)   # torch.Size([2, 2])  inner 3 cancels

</details>

**Problem 2.** Type this in Colab. Broadcast a column against a row. Build col = torch.tensor([[10.],[20.],[30.]]) (shape (3,1)) and row = torch.tensor([[1.,2.,3.,4.]]) (shape (1,4)), add them, and print the result and its shape. Predict the shape first.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Add the (3,1) and (1,4) tensors directly. — _Broadcasting stretches each size-1 axis to meet the other operand._
- Print .shape to verify it is (3,4). — _Cell (i,j) = col[i] + row[j]; printing shape catches a silent mis-broadcast._

**Answer:** col = torch.tensor([[10.], [20.], [30.]])
row = torch.tensor([[1., 2., 3., 4.]])
grid = col + row
print(grid.shape)   # torch.Size([3, 4])
print(grid)
# tensor([[11., 12., 13., 14.],
#         [21., 22., 23., 24.],
#         [31., 32., 33., 34.]])

</details>

**Problem 3.** Type this in Colab. The silent-broadcast pitfall. You meant to add two length-3 vectors, but one is a row and one is a column: add torch.tensor([1., 2., 3.]) (shape (3,)) to torch.tensor([[10.],[20.],[30.]]) (shape (3,1)). Print the result shape — it is NOT (3,). Explain in a comment what happened.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Add the (3,) and (3,1) tensors and print .shape. — _Broadcasting aligns trailing dims, turning this into a (3,3) grid — a wrong answer with no error._
- Note the fix: make both the same shape (e.g. flatten the column). — _Printing shapes is the only way to catch a silent mis-broadcast._

**Answer:** v   = torch.tensor([1., 2., 3.])          # (3,)
col = torch.tensor([[10.], [20.], [30.]]) # (3,1)
out = v + col
print(out.shape)   # torch.Size([3, 3])  -- a 3x3 grid, NOT (3,)!
print(out)
# tensor([[11., 12., 13.],
#         [21., 22., 23.],
#         [31., 32., 33.]])
# Fix: align shapes, e.g. v + col.flatten() -> tensor([11., 22., 33.])

</details>

**Problem 4.** Type this in Colab. Reduce over a chosen axis with and without keepdim. Make g = torch.tensor([[1.,2.,3.,4.],[5.,6.,7.,8.],[9.,10.,11.,12.]]). Print g.sum(dim=1) and its shape, then g.sum(dim=1, keepdim=True) and its shape.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Sum across columns with dim=1. — _dim is the axis that collapses; dim=1 leaves one value per row._
- Add keepdim=True to retain the axis as size 1. — _Keeping it as (3,1) lets the result broadcast back (the softmax-normalize trick)._

**Answer:** g = torch.tensor([[1.,2.,3.,4.],[5.,6.,7.,8.],[9.,10.,11.,12.]])
print(g.sum(dim=1))                 # tensor([10., 26., 42.])
print(g.sum(dim=1).shape)           # torch.Size([3])
print(g.sum(dim=1, keepdim=True))   # tensor([[10.],[26.],[42.]])
print(g.sum(dim=1, keepdim=True).shape)  # torch.Size([3, 1])

</details>

**Problem 5.** Type this in Colab. A row-softmax in three lines. Take g from the previous task. Compute e = torch.exp(g - g.max(dim=1, keepdim=True).values), then probs = e / e.sum(dim=1, keepdim=True). Print probs.shape and probs.sum(dim=1) to confirm each row sums to 1.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Subtract the per-row max with keepdim=True before exp. — _Numerically stabilizes the exponent; keepdim keeps the (3,1) shape so it broadcasts back._
- Divide by the per-row sum (also keepdim=True). — _Normalizes each row to a probability distribution summing to 1._

**Answer:** e = torch.exp(g - g.max(dim=1, keepdim=True).values)
probs = e / e.sum(dim=1, keepdim=True)
print(probs.shape)         # torch.Size([3, 4])
print(probs.sum(dim=1))    # tensor([1., 1., 1.])

</details>

**Problem 6.** Type this in Colab. A linear layer plus argmax prediction. Make a batch x = torch.randn(4, 5) (seed first) and a weight W = torch.randn(5, 3). Compute z = x @ W, print z.shape, then pred = z.argmax(dim=-1) and print pred and pred.shape.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Matmul (4,5) @ (5,3) to get scores (4,3). — _Every linear layer is a matmul; inner 5 cancels, leaving 3 class scores per example._
- Take argmax(dim=-1) over the class axis. — _It returns the index of the top score per row — the predicted class; reducing dim=0 would wrongly collapse the batch._

**Answer:** torch.manual_seed(0)
x = torch.randn(4, 5)
W = torch.randn(5, 3)
z = x @ W
print(z.shape)              # torch.Size([4, 3])
pred = z.argmax(dim=-1)
print(pred)                # tensor([0, 1, 0, 1])
print(pred.shape)          # torch.Size([4])

</details>

**Problem 7.** Type this in Colab. The view-vs-reshape pitfall. Make x = torch.arange(24).reshape(2, 3, 4) and xp = x.permute(0, 2, 1). Try xp.view(-1) in a try/except and print the failure, then show xp.reshape(-1) works and print its shape.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Call xp.view(-1) on the permuted (non-contiguous) tensor. — _view needs contiguous row-major memory; permute only reorders strides, so it raises._
- Fall back to xp.reshape(-1) (or xp.contiguous().view(-1)). — _reshape copies into a fresh contiguous block when needed._

**Answer:** x = torch.arange(24).reshape(2, 3, 4)
xp = x.permute(0, 2, 1)              # (2,4,3), non-contiguous
print(xp.is_contiguous())           # False
try:
    xp.view(-1)
except RuntimeError as err:
    print("view failed:", "not compatible" in str(err))  # view failed: True
print(xp.reshape(-1).shape)         # torch.Size([24])  reshape copies

</details>

**Problem 8.** Type this in Colab. Reshape moves: build x = torch.arange(24).reshape(2, 3, 4), then print the shapes of x.flatten(1), x.unsqueeze(0), torch.cat([x, x], dim=0), and torch.stack([x, x], dim=0). Predict each before running.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use flatten(1) and unsqueeze(0) to merge and insert axes. — _flatten(1) merges axes from dim 1 on; unsqueeze(0) inserts a size-1 batch axis._
- Compare cat (joins an existing axis) with stack (adds a new one). — _cat on dim 0 doubles that axis; stack adds a fresh leading axis of size 2._

**Answer:** x = torch.arange(24).reshape(2, 3, 4)
print(x.flatten(1).shape)               # torch.Size([2, 12])
print(x.unsqueeze(0).shape)             # torch.Size([1, 2, 3, 4])
print(torch.cat([x, x], dim=0).shape)   # torch.Size([4, 3, 4])
print(torch.stack([x, x], dim=0).shape) # torch.Size([2, 2, 3, 4])

</details>